# Diagrams and Schematics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dtabuena/Resources/blob/main/Art/Diagrams_and_Schematics.ipynb)

Cartoon and schematic recipes for figures and graphical abstracts. Every section runs on random stand-in data, so nothing here depends on a real dataset.

Shared setup runs once below. Each schematic after that is self-contained.

In [ ]:
import warnings
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

response = urllib.request.urlretrieve('https://raw.githubusercontent.com/dtabuena/Resources/main/Matplotlib_Config/drt_mpl_config.py', 'drt_mpl_config.py')
%run drt_mpl_config.py

## 96-Well Plate Cartoon

One plate, full circles, black outlines. Green well passed, orange well failed, at a random 10 percent failure rate.

Text-stripped relative to the source figure: tick labels (A-H, 1-12), plate title, legend, and the summary prints are all gone.

In [ ]:
# 96 wells
EMPTY_COLOR = 'lightgray'         # was #CCCCCC, unused while the plate is full
SUCCESS_COLOR = 'mediumseagreen'  # was #4CAF50, green  - passed
FAILURE_COLOR = 'darkorange'      # was #F57C00, orange - failed
OUTLINE_COLOR = 'k'

ROW_LABELS = list('ABCDEFGH')
N_ROWS = 8
N_COLS = 12
WELL_RADIUS = 0.45

FAILURE_RATE = 0.10
RANDOM_SEED = 0

In [ ]:
"""Random stand-in for well_status: one row per well, pass or fail."""

random_generator = np.random.default_rng(RANDOM_SEED)

all_wells = [f'{row_letter}{col_number:02d}'
             for row_letter in ROW_LABELS
             for col_number in range(1, N_COLS + 1)]

well_status = pd.DataFrame({
    'well': all_wells,
    'passed': random_generator.random(len(all_wells)) >= FAILURE_RATE,
})

display(well_status.head())

In [ ]:
def well_to_rc(well_name):
    """Convert 'A01' -> (row_index, col_index) zero-based."""
    row_letter = well_name[0]
    if row_letter not in ROW_LABELS:
        raise ValueError(f"Unrecognized row letter '{row_letter}' in well '{well_name}'")
    row_index = ROW_LABELS.index(row_letter)
    col_index = int(well_name[1:]) - 1
    return row_index, col_index


def draw_well(target_axis, col_index, row_index, face_color):
    """One full circle per well. Linewidth comes from the config rcParams."""
    target_axis.add_patch(mpl.patches.Circle((col_index, -row_index), WELL_RADIUS,
                                             facecolor=face_color,
                                             edgecolor=OUTLINE_COLOR))


def plot_plate(target_axis, status_frame):
    for _row_label, well_row in status_frame.iterrows():
        try:
            row_index, col_index = well_to_rc(well_row['well'])
        except (ValueError, IndexError) as parse_error:
            warnings.warn(f"Could not parse well '{well_row['well']}', "
                          f"skipping: {parse_error}")
            continue
        face_color = SUCCESS_COLOR if well_row['passed'] else FAILURE_COLOR
        draw_well(target_axis, col_index, row_index, face_color)

    target_axis.set_xlim(-0.8, N_COLS - 0.2)
    target_axis.set_ylim(-(N_ROWS - 0.2), 0.8)
    target_axis.set_xticks([])
    target_axis.set_yticks([])
    target_axis.set_aspect('equal')

In [ ]:
fig, panel_axis = plt.subplots(figsize=(2.25, 1.5))

plot_plate(panel_axis, well_status)

plt.tight_layout()
plt.show()